In [ ]:

!pip install lightgbm optuna scikit-learn pandas numpy matplotlib seaborn joblib tensorflow


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 404.7/404.7 kB 12.8 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import optuna

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

import lightgbm as lgb

import tensorflow as tf
from tensorflow.keras import layers, models

from datetime import timedelta

In [ ]:

from google.colab import files
uploaded = files.upload()


!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

print("Kaggle setup done!")



KeyboardInterrupt: 

In [ ]:
!wget https://archive.ics.uci.edu/static/public/352/online+retail.zip
!unzip "online retail.zip"


In [ ]:
!unzip "online+retail.zip"


In [ ]:
import pandas as pd

df = pd.read_excel("Online Retail.xlsx")
df.head()


In [ ]:
df_clean = df.copy()

# Remove missing StockCodes, negative quantities (returns), etc.
df_clean = df_clean.dropna(subset=["StockCode", "InvoiceDate"])
df_clean = df_clean[df_clean["Quantity"] > 0]

# Convert InvoiceDate to datetime
df_clean["InvoiceDate"] = pd.to_datetime(df_clean["InvoiceDate"])

# Extract only required columns
df_clean = df_clean[["StockCode", "Quantity", "InvoiceDate"]]

# Create a Date column (without time)
df_clean["Date"] = df_clean["InvoiceDate"].dt.date

# Group by StockCode + Date to get daily total quantity sold
daily_demand = df_clean.groupby(["StockCode", "Date"])["Quantity"].sum().reset_index()

daily_demand["Date"] = pd.to_datetime(daily_demand["Date"])
daily_demand = daily_demand.sort_values(["StockCode", "Date"])

daily_demand.head()

In [ ]:
daily_demand["StockCode"].value_counts().head(20)


In [ ]:
# Step 1: Normalize column names
daily_demand.columns = [col.lower() for col in daily_demand.columns]

# Step 2: Pick SKU
sku = "85123A"

sku_df = daily_demand[daily_demand["stockcode"] == sku].copy()

print("Total rows for selected SKU:", len(sku_df))
display(sku_df.head())

# Step 3: Convert to datetime
sku_df["date"] = pd.to_datetime(sku_df["date"], errors="coerce")
sku_df = sku_df.dropna(subset=["date"])

# Step 4: Sort
sku_df = sku_df.sort_values("date")

# Step 5: Reindex for continuous dates
full_dates = pd.date_range(sku_df["date"].min(), sku_df["date"].max())

# IMPORTANT FIX: set index BEFORE dropping stockcode
sku_df = sku_df.set_index("date")

# Fill missing dates
sku_df = sku_df.reindex(full_dates, fill_value=0)

# Drop the stockcode column if still present
if "stockcode" in sku_df.columns:
    sku_df = sku_df.drop(columns=["stockcode"])

# Reset index
sku_df = sku_df.reset_index()

# Rename columns cleanly
sku_df.columns = ["date", "quantity"]

print("Final cleaned time series:")
display(sku_df.head(10))
display(sku_df.tail(10))


In [ ]:
df_feat = sku_df.copy()

# --- Lag Features ---
# lag_1 = kal ka demand
# lag_7 = pichle hafte ka demand
# lag_14 = pichle 2 hafte ka demand
df_feat["lag_1"] = df_feat["quantity"].shift(1)
df_feat["lag_7"] = df_feat["quantity"].shift(7)
df_feat["lag_14"] = df_feat["quantity"].shift(14)

# --- Rolling Window Features ---
# roll_mean_7 = pichle 7 din ka average (trend + seasonality capture karta hai)
# roll_std_7  = pichle 7 din ka variation (kitna fluctuation ho raha hai)
df_feat["roll_mean_7"] = df_feat["quantity"].rolling(window=7).mean()
df_feat["roll_std_7"] = df_feat["quantity"].rolling(window=7).std()

# --- Target Feature ---
# target_7d = aage ke 7 din ka combined demand
# Yeh forecasting ka main label hai
df_feat["target_7d"] = df_feat["quantity"].shift(-1).rolling(7).sum()

# Remove rows with NaN after shifting windows
df_feat = df_feat.dropna()

print("Feature engineering sample rows:")
df_feat.head()

In [ ]:

cutoff = df_feat["date"].iloc[-40]

train = df_feat[df_feat["date"] <= cutoff]
test  = df_feat[df_feat["date"] > cutoff]

features = ["lag_1", "lag_7", "lag_14", "roll_mean_7", "roll_std_7"]

X_train = train[features]
y_train = train["target_7d"]

X_test = test[features]
y_test = test["target_7d"]

print("Train shape:", X_train.shape)
print("Test shape :", X_test.shape)


RandomForest

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
import numpy as np

rf = RandomForestRegressor(
    n_estimators=300,
    random_state=42,
    max_depth=None
)

rf.fit(X_train, y_train)

rf_preds = rf.predict(X_test)
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_preds))

print("RF RMSE:", rf_rmse)


LightGBM

In [ ]:
import lightgbm as lgb

train_data = lgb.Dataset(X_train, label=y_train)

params = {
    "objective": "regression",
    "metric": "rmse",
    "learning_rate": 0.05,
    "num_leaves": 40,
    "feature_fraction": 0.8
}

lgb_model = lgb.train(params, train_data, num_boost_round=300)

lgb_preds = lgb_model.predict(X_test)
lgb_rmse = np.sqrt(mean_squared_error(y_test, lgb_preds))

print("LightGBM RMSE:", lgb_rmse)


In [ ]:
import numpy as np

def make_lstm_sequences(X, y, seq_len=1):
    Xs, ys = [], []
    for i in range(len(X) - seq_len):
        Xs.append(X.iloc[i : i + seq_len].values)
        ys.append(y.iloc[i + seq_len])
    return np.array(Xs), np.array(ys)

# Ab train aur test ke sequences banate hain
X_train_lstm, y_train_lstm = make_lstm_sequences(X_train, y_train)
X_test_lstm, y_test_lstm = make_lstm_sequences(X_test, y_test)

print("LSTM Train shape:", X_train_lstm.shape)
print("LSTM Test shape :", X_test_lstm.shape)

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

model = models.Sequential([
    layers.LSTM(64, activation='tanh', input_shape=(X_train_lstm.shape[1], X_train_lstm.shape[2])),
    layers.Dense(32, activation='relu'),
    layers.Dense(1)
])

model.compile(optimizer='adam', loss='mse')

history = model.fit(
    X_train_lstm, y_train_lstm,
    validation_split=0.1,
    epochs=25,        # Increased epochs for better learning
    batch_size=16,
    verbose=1
)

# Plot loss curve
import matplotlib.pyplot as plt

plt.plot(history.history['loss'], label='train_loss')
plt.plot(history.history['val_loss'], label='val_loss')
plt.legend()
plt.title("LSTM Loss Curve")
plt.show()

In [ ]:
from sklearn.metrics import mean_squared_error
import numpy as np

lstm_preds = model.predict(X_test_lstm).reshape(-1)
lstm_rmse = np.sqrt(mean_squared_error(y_test_lstm, lstm_preds))

print("LSTM RMSE:", lstm_rmse)
